# Zero-to-Hero: Production AI Reasoning Agent

This notebook walks through a production-style reasoning agent framework built with LangGraph-compatible architecture, modular tools, memory, observability, and evaluation.

## 1. Why AI Agents (Beyond Chat)

A production agent must do more than next-token generation. It should:
- Decompose tasks into steps
- Choose tools dynamically
- Observe outcomes and revise plan
- Store memory across turns
- Surface telemetry and traces for debugging

## 2. Architecture Overview

Core components:
- Planner
- Tool Router
- Executor
- Observation Processor
- Reflector
- Response Generator
- Error Handler
- Execution Logger

Primary runtime includes fallback-safe execution path for restricted environments.

In [1]:
from pathlib import Path
import asyncio
import os

from reasoning_agent.config import get_settings
from reasoning_agent.agent.runner import AgentRunner
from reasoning_agent.agent.models import AgentRequest
from reasoning_agent.constants import BENCHMARK_DATA_PATH

## 3. Configure Runtime

For notebook reproducibility, we enable offline-safe mode when local model server is unavailable.

In [2]:
settings = get_settings()
settings.memory.chroma_enabled = False
settings.agent.graph_timeout_seconds = 1

# Set fallback-safe execution for notebook environments without Ollama daemon.
settings.agent.use_llm_for_planning = False
settings.agent.use_llm_for_response = False
os.environ['AGENT_OFFLINE_MODE'] = '1'

runner = AgentRunner(settings=settings)
settings

## 4. ReAct-style Execution Example

Run one multi-step prompt and inspect plan/tool/observation trace.

In [3]:
async def run_query(prompt: str):
    return await runner.run(AgentRequest(query=prompt, session_id='notebook'))

result = await run_query('Convert 12 kilometers to miles and explain method')
result

## 5. Inspect Trace and Telemetry

The result includes plan, tool calls, observations, reflection note, and latency metrics.

In [4]:
print('Answer:', result.answer)
print('Plan steps:', len(result.plan))
print('Tool calls:', len(result.tool_calls))
print('Metrics:', result.metrics)

Answer: Based on available observations: unit_converter: {'converted_value': 7.456454306848007, 'tool': 'unit_converter', 'latency_ms': 0.01386601070407778}
Plan steps: 1
Tool calls: 1
Metrics: {'total_latency_ms': 7.099355992977507, 'tool_calls': 1.0, 'iterations': 1.0, 'tool.unit_converter.success': 1.0}


## 6. Memory, Tooling, and Safety

Safety controls included in framework:
- Safe calculator parser (no eval)
- Restricted Python runner with timeout and memory limits
- File access bounded to workspace root
- Pydantic schema validation for tool contracts and parsed outputs

## 7. Full Benchmark Suite (120 prompts)

Benchmark categories: math, reasoning, knowledge, current events, tool use, multi-step, code generation, document QA.
This cell executes the benchmark script and writes reports + Plotly dashboards to `artifacts/benchmarks/`.

In [5]:
import subprocess
import sys

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

cmd = [sys.executable, str(project_root / 'scripts' / 'run_benchmarks.py')]
completed = subprocess.run(cmd, capture_output=True, text=True, check=False, cwd=project_root)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)

Benchmark complete
predictions_jsonl: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/AI Agents/Production AI Reasoning Agent/artifacts/benchmarks/predictions_20260628T182224Z.jsonl
summary_json: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/AI Agents/Production AI Reasoning Agent/artifacts/benchmarks/summary_20260628T182224Z.json
summary_csv: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/AI Agents/Production AI Reasoning Agent/artifacts/benchmarks/summary_20260628T182224Z.csv
summary_parquet: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/AI Agents/Production AI Reasoning Agent/artifacts/benchmarks/summary_20260628T182224Z.parquet



## 8. Load Benchmark Summary Table

In [6]:
import polars as pl

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

bench_dir = project_root / 'artifacts' / 'benchmarks'
csv_files = sorted(bench_dir.glob('summary_*.csv'))
if not csv_files:
    raise RuntimeError(f'No benchmark summary files found in {bench_dir}')
latest_csv = csv_files[-1]
summary_df = pl.read_csv(latest_csv)
summary_df

## 9. Observability Artifacts

Generated assets:
- JSONL predictions
- JSON / CSV / Parquet summary
- Plotly HTML dashboards (latency, success rate, tool usage, radar)
- Run traces in `logs/agent_runs.jsonl`

In [7]:
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

for path in sorted((project_root / 'artifacts' / 'benchmarks').glob('*')):
    print(path)

/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/AI Agents/Production AI Reasoning Agent/artifacts/benchmarks/latency.html
/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/AI Agents/Production AI Reasoning Agent/artifacts/benchmarks/predictions_20260627T115014Z.jsonl
/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/AI Agents/Production AI Reasoning Agent/artifacts/benchmarks/predictions_20260627T115032Z.jsonl
/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/AI Agents/Production AI Reasoning Agent/artifacts/benchmarks/predictions_20260627T115304Z.jsonl
/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/AI Agents/Production AI Reasoning Agent/artifacts/benchmarks/predictions_20260627T120100Z.jsonl
/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/AI Agents/Production AI Reasoning Agent/artifacts/benchmarks/predictions_20260627T120120Z.jsonl
/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/AI Agents/Production AI Reasoning Agent/artifacts/benchmarks/predict

## 10. Next Steps

To switch from fallback mode to full local-model mode:
1. Start Ollama daemon and pull benchmark models
2. Enable `use_llm_for_planning` and `use_llm_for_response` in config
3. Disable `AGENT_OFFLINE_MODE`
4. Re-run benchmark for live model + live retrieval evaluation